In [1]:
from kafka import KafkaProducer
import pandas as pd
from models import json_serializer, ride_from_row
import dataclasses
import time

In [2]:
url = "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-10.parquet"
raw_df = pd.read_parquet(url)

In [3]:
raw_df[raw_df.tip_amount < 0]

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,...,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee
18772,2,2025-10-14 14:34:40,2025-10-14 14:34:49,N,1.0,193,193,1.0,0.0,-3.0,...,-0.5,-0.9,0.0,NaN,-1.0,-5.4,3.0,1.0,0.0,0.0
22893,2,2025-10-17 09:13:29,2025-10-17 09:13:33,N,1.0,193,193,1.0,0.0,-3.0,...,-0.5,-0.9,0.0,NaN,-1.0,-5.4,3.0,1.0,0.0,0.0
24807,2,2025-10-18 15:30:54,2025-10-18 15:31:02,N,1.0,193,193,6.0,0.0,-3.0,...,-0.5,-0.9,0.0,NaN,-1.0,-5.4,3.0,1.0,0.0,0.0


In [3]:
server = "localhost:9092"
producer = KafkaProducer(
    bootstrap_servers=[server], 
    value_serializer=json_serializer
)

In [24]:
# There are numpy types in the ride dataclass that are not JSON serializable,
# so its necessary to convert them to native python types, which is what the numpy_encoder function does.
topic_name = 'rides'
first_ride = ride_from_row(raw_df.iloc[1])
producer.send(topic_name, dataclasses.asdict(first_ride))
producer.flush()

In [ ]:
# topic_name = 'green_taxi_rides'
t0 = time.time()
for _, row in raw_df.iterrows():
    if row.passenger_count.isna():
        continue
    ride = ride_from_row(row)
    producer.send('green-trips', dataclasses.asdict(ride))
producer.flush()
t1 = time.time()
print(f"Time taken: {t1 - t0} seconds")

Time taken: 19.878994703292847 seconds


In [1]:
raw_df.shape

NameError: name 'raw_df' is not defined